### Purpose
Core payment volume and channel analysis.

**Data:** `marts.fct_transactions` + `dim_users` + `dim_merchants` (Postgres). Requires `make docker-up` + `dbt build`.

**Charts:** `notebooks/charts/transactional/`

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(BASE_DIR))

from notebooks.utils.data_loader import get_merged_data, print_data_sanity
from notebooks.utils.style import C, PAYMENT_COLORS, STATUS_COLORS, INR_CR, apply_theme, style_axes
from notebooks.utils.visualization import save_chart, new_fig

apply_theme()
CHART_SUBDIR = "transactional"

df, data_source = get_merged_data()
df["success_amount"] = df["amount"].where(df["is_success"], 0)
print_data_sanity(df, data_source)

**Metrics** — Attempt volume = sum of all `amount`; Success GMV = `amount` where `status = success` only.

In [ ]:
attempt_vol = df["amount"].sum()
success_gmv = df["success_amount"].sum()
print(f"Attempt volume : ₹{attempt_vol:,.0f}")
print(f"Success GMV     : ₹{success_gmv:,.0f}")
if attempt_vol > 0:
    print(f"Capture rate    : {success_gmv / attempt_vol:.1%}")
else:
    print("Capture rate    : N/A (no attempts)")

### Chart - Monthly volume & payment success rate

In [ ]:
m = df.groupby("month").agg(
    attempt_volume=("amount", "sum"),
    success_gmv=("success_amount", "sum"),
    success_rate=("is_success", "mean"),
).reset_index()

fig, ax1 = plt.subplots(figsize=(11, 4.5), layout="constrained")
x = np.arange(len(m))
w = 0.36
ax1.bar(x - w / 2, m["attempt_volume"], w, color=C["primary"], label="Attempt volume")
ax1.bar(x + w / 2, m["success_gmv"], w, color=C["success"], label="Success GMV")
ax1.set_ylabel("INR")
ax1.yaxis.set_major_formatter(INR_CR)
tick_idx = x[::2]
ax1.set_xticks(tick_idx)
ax1.set_xticklabels(m["month"].iloc[::2])
style_axes(ax1)

ax2 = ax1.twinx()
ax2.plot(x, m["success_rate"] * 100, color=C["secondary"], marker="o", markersize=4, linewidth=1.6, label="Success %")
ax2.set_ylabel("Success rate (%)")
ax2.set_ylim(m["success_rate"].min() * 100 - 1, m["success_rate"].max() * 100 + 1)
ax2.spines["top"].set_visible(False)

h1, l1 = ax1.get_legend_handles_labels()
h2, l2 = ax2.get_legend_handles_labels()
ax1.legend(h1 + h2, l1 + l2, loc="upper left", ncol=3, fontsize=8)
ax1.set_title("Monthly volume vs payment success rate", loc="left", fontweight="600")
save_chart(fig, "monthly_attempt_volume_vs_success_gmv", CHART_SUBDIR)

`Volume scales while success rate stays flat — growth is attempt-driven, not captured-revenue driven.`

### Chart - Transaction status mix

In [ ]:
order = ["success", "failed", "declined", "disputed"]
counts = df["status"].value_counts().reindex(order)
fig, ax = new_fig(9, 4)
bars = ax.barh(counts.index, counts.to_numpy(), color=[STATUS_COLORS[s] for s in counts.index], height=0.55)
ax.set_xlabel("Transactions")
ax.set_title("Status mix (all attempts)", loc="left", fontweight="600")
ax.invert_yaxis()
style_axes(ax, grid_axis="x")
save_chart(fig, "status_mix", CHART_SUBDIR)

`~43% of attempts are non-success — ops monitors the failed / declined / disputed split for routing and issuer issues.`

### Chart - Daily payment success rate

In [ ]:
daily = df.groupby(df["transaction_ts"].dt.date).agg(success_rate=("is_success", "mean")).reset_index()
daily["pct"] = daily["success_rate"] * 100
daily["ma7"] = daily["pct"].rolling(7, min_periods=1).mean()
platform = df["is_success"].mean() * 100

fig, ax = new_fig(11, 4)
ax.fill_between(daily["transaction_ts"], daily["pct"], alpha=0.12, color=C["primary"])
ax.plot(daily["transaction_ts"], daily["pct"], color=C["primary"], linewidth=0.8, alpha=0.35, label="Daily")
ax.plot(daily["transaction_ts"], daily["ma7"], color=C["secondary"], linewidth=2.2, label="7-day avg")
ax.axhline(platform, color=C["neutral"], linestyle="--", linewidth=1.2, label=f"Platform avg ({platform:.1f}%)")
ax.set_ylabel("Payment success rate (%)")
ax.set_ylim(max(40, daily["pct"].min() - 3), min(80, daily["pct"].max() + 3))
ax.legend(loc="lower right", fontsize=8)
ax.set_title("Daily payment success rate (status = success)", loc="left", fontweight="600")
style_axes(ax)
save_chart(fig, "daily_success_rate", CHART_SUBDIR)

`Derived from `status` — no separate auth column. The 7-day line is what ops tracks; daily noise is normal at txn-day granularity.`

### Chart - Payment method mix (txn share)

In [ ]:
mix = df.groupby(["month", "payment_method"])["transaction_id"].count().unstack(fill_value=0)
mix_pct = mix.div(mix.sum(axis=1), axis=0) * 100

fig, ax = new_fig(11, 4.5)
for method in mix_pct.columns:
    ax.plot(
        mix_pct.index, mix_pct[method], marker="o", markersize=3, linewidth=1.8,
        color=PAYMENT_COLORS.get(method, C["neutral"]), label=method,
    )
ax.set_ylabel("Share of transactions (%)")
ax.set_ylim(mix_pct.min().min() - 1, mix_pct.max().max() + 1)
ax.legend(loc="center left", bbox_to_anchor=(1.01, 0.5), fontsize=8)
ax.set_title("Payment method share by month", loc="left", fontweight="600")
plt.setp(ax.get_xticklabels(), rotation=0)
style_axes(ax)
save_chart(fig, "payment_method_mix_over_time", CHART_SUBDIR)

`Line view shows each rail's share clearly — in this synthetic set mix is stable (~20% each); production teams watch UPI drift here.`

### Chart - Payment success rate by method

In [ ]:
sr = df.groupby("payment_method").agg(rate=("is_success", "mean"), n=("transaction_id", "count"))
sr = sr[sr["n"] >= 100].sort_values("rate")
platform = df["is_success"].mean() * 100
rates_pct = (sr["rate"] * 100).to_numpy()

from matplotlib.transforms import blended_transform_factory

fig, ax = new_fig(9, 4)
bars = ax.barh(sr.index, rates_pct, color=C["primary"], height=0.55)
ax.axvline(platform, color=C["neutral"], linestyle="--", linewidth=1.2, zorder=3)
ax.set_xlim(rates_pct.min() - 0.35, rates_pct.max() + 0.6)
label_transform = blended_transform_factory(ax.transData, ax.transAxes)
ax.annotate(
    f"Platform {platform:.1f}%",
    xy=(platform, 0.04), xytext=(6, 0), textcoords="offset points",
    xycoords=label_transform, ha="left", va="bottom", fontsize=8, color=C["neutral"],
)
ax.set_xlabel("Payment success rate (%)")
ax.set_title("Success rate by payment method", loc="left", fontweight="600")
style_axes(ax, grid_axis="x")
save_chart(fig, "success_rate_by_payment_method", CHART_SUBDIR)

`All methods cluster near platform avg in synthetic data — in production, material gaps vs the dashed line trigger routing reviews.`

### Chart - Fail + decline rate by method

In [ ]:
fd = df.groupby("payment_method").agg(
    total=("transaction_id", "count"),
    bad=("status", lambda s: s.isin(["failed", "declined"]).sum()),
)
fd = fd[fd["total"] >= 100].assign(rate=lambda d: d["bad"] / d["total"] * 100).sort_values("rate")

fig, ax = new_fig(9, 4)
bars = ax.barh(fd.index, fd["rate"].to_numpy(), color=C["danger"], height=0.55, alpha=0.9)
ax.set_xlabel("Failed + declined (%)")
ax.set_title("Failure & decline rate by method", loc="left", fontweight="600")
style_axes(ax, grid_axis="x")
save_chart(fig, "fail_decline_rate_by_payment_method", CHART_SUBDIR)

`Complements success rate — methods above ~30% combined fail+decline warrant issuer-specific investigation.`

### Chart - Monthly average ticket value (ATV)

In [ ]:
atv = df.groupby("month").agg(
    attempt_atv=("amount", "mean"),
    success_atv=("success_amount", lambda s: s[s > 0].mean()),
).reset_index()

fig, ax = new_fig(11, 4)
ax.plot(atv["month"], atv["attempt_atv"], color=C["primary"], marker="o", markersize=4, linewidth=1.8, label="Attempt ATV")
ax.plot(atv["month"], atv["success_atv"], color=C["success"], marker="o", markersize=4, linewidth=1.8, label="Success ATV")
ax.set_ylabel("INR / transaction")
ax.yaxis.set_major_formatter(INR_CR)
ax.legend(loc="upper left", fontsize=8)
ax.set_title("Monthly average ticket value", loc="left", fontweight="600")
plt.setp(ax.get_xticklabels(), rotation=0)
style_axes(ax)
save_chart(fig, "monthly_atv", CHART_SUBDIR)

`Success ATV above attempt ATV means failed txns skew smaller — typical when high-ticket attempts fail more often.`